In [1]:
import os
import pickle
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ==============================================================================
# EXPERIMENT CONFIGURATION
# ==============================================================================

EXPERIMENT_MODE = "BASELINE"
#EXPERIMENT_MODE = "PRETRAINED_6CH"
# Options:
#   "BASELINE"       -> 4 channels, no pretraining
#   "PRETRAINED_6CH" -> 6 channels, pretrained CNN


CONFIGS = {

    # ======================================================================
    # OPTION 1: ORIGINAL / BASELINE
    # 4 channels, no pretrained model
    # ======================================================================

    "BASELINE": {
        "n_channels": 4,
        "use_pretraining": False,

        "channel_names": [
            "fhr",
            "uc",
            "fhr_derivative",
            "uc_derivative",
        ],

        "pretrained_model_path": None,
    },


    # ======================================================================
    # OPTION 2: PRETRAINED
    # 6 channels = original 4 + ACC + DEC
    # ======================================================================

    "PRETRAINED_6CH": {
        "n_channels": 6,
        "use_pretraining": True,

        "channel_names": [
            "fhr",
            "uc",
            "fhr_derivative",
            "uc_derivative",
            "acc",
            "dec",
        ],

        "pretrained_model_path": (
            "/content/drive/My Drive/Colab Notebooks/"
            "Fetal_project/"
            "15min_5min_acc_dec_detector.pt"
        ),
    }
}


CONFIG = CONFIGS[EXPERIMENT_MODE]

print("=" * 80)
print("EXPERIMENT CONFIGURATION")
print("=" * 80)
print(f"Mode             : {EXPERIMENT_MODE}")
print(f"Channels         : {CONFIG['n_channels']}")
print(f"Pretraining      : {CONFIG['use_pretraining']}")
print(f"Channel names    : {CONFIG['channel_names']}")
print(f"Pretrained model : {CONFIG['pretrained_model_path']}")

EXPERIMENT CONFIGURATION
Mode             : BASELINE
Channels         : 4
Pretraining      : False
Channel names    : ['fhr', 'uc', 'fhr_derivative', 'uc_derivative']
Pretrained model : None


In [3]:
def create_mil_bags(
    cache_path,
    pretrain_model=None,
    n_channels=4,
    window_size=3600,
    stride_size=1200,
    max_allowable_nan_pct=0.60,
    device="cpu"
):

    cache = np.load(
        cache_path,
        allow_pickle=True
    )

    record_ids = [
        key.replace("rec_", "")
        for key in cache.files
        if key.startswith("rec_")
    ]

    mil_bags = {}

    for rid in record_ids:

        signal_matrix = cache[f"rec_{rid}"]

        total_samples = signal_matrix.shape[1]

        valid_record_chunks = []

        for start_idx in range(
            0,
            total_samples - window_size + 1,
            stride_size
        ):

            end_idx = start_idx + window_size

            window = signal_matrix[:, start_idx:end_idx]

            # --------------------------------------------------------------
            # Always start from the original 4 CTG channels
            # --------------------------------------------------------------

            if window.shape[0] < 4:
                continue

            window = window[:4]

            # --------------------------------------------------------------
            # Existing quality filtering
            # --------------------------------------------------------------

            if np.isnan(window[2]).mean() > max_allowable_nan_pct:
                continue

            if np.isnan(window[3]).mean() > max_allowable_nan_pct:
                continue

            # ------------------------------------------------------------------------------
            # Safety Check: Enforce 4 input channels & semantics
            # ------------------------------------------------------------------------------
            assert window.shape[0] == 4, f"Expected 4 channels, got {window.shape[0]}"

            # Semantics check reminder:
            # window[0] = FHR
            # window[1] = UC
            # window[2] = FHR missingness
            # window[3] = UC missingness

            # --------------------------------------------------------------
            # BASELINE
            # --------------------------------------------------------------

            if n_channels == 4:

                window_final = window

            # --------------------------------------------------------------
            # PRETRAINED 6-CHANNEL
            # --------------------------------------------------------------

            elif n_channels == 6:

                if pretrain_model is None:
                    raise ValueError(
                        "6-channel mode requires a pretrained model."
                    )

                win_tensor = (
                    torch.from_numpy(window)
                    .float()
                    .unsqueeze(0)
                    .to(device)
                )

                with torch.no_grad():

                    pretrained_output = pretrain_model(
                        win_tensor
                    )

                # Keep ACC + DEC only
                acc_dec = (
                    #pretrained_output[:, 1:3, :]
                    pretrained_output[:, [0,2], :]
                    .squeeze(0)
                    .cpu()
                    .numpy()
                )

                window_final = np.concatenate(
                    [window, acc_dec],
                    axis=0
                )

            else:

                raise ValueError(
                    f"Unsupported n_channels={n_channels}"
                )

            # --------------------------------------------------------------
            # Safety check
            # --------------------------------------------------------------

            if window_final.shape != (
                n_channels,
                window_size
            ):

                raise RuntimeError(
                    f"Unexpected window shape: "
                    f"{window_final.shape}. "
                    f"Expected ({n_channels}, {window_size})"
                )

            valid_record_chunks.append(
                window_final
            )

        if valid_record_chunks:

            mil_bags[int(rid)] = np.stack(
                valid_record_chunks,
                axis=0
            )

    return mil_bags

In [4]:
# ==============================================================================
# 1. DEVICE AND PATHS
# ==============================================================================

FS = 4
FULL_CACHE_PATH = "/content/drive/My Drive/Colab Notebooks/Fetal_project/ctu_full_length_signals_cache.npz"
PRETRAIN_MODEL_PATH = "/content/drive/My Drive/Colab Notebooks/Fetal_project/15min_5min_acc_dec_detector.pt"

WINDOW_MINUTES = 15
STRIDE_MINUTES = 5

WINDOW_SIZE = WINDOW_MINUTES * 60 * FS
STRIDE_SIZE = STRIDE_MINUTES * 60 * FS

MAX_ALLOWABLE_NAN_PCT = 0.90 #0.3 #0.60 #

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"[DEVICE] Using: {device}")
print(f"[CONFIG] Window size: {WINDOW_SIZE} samples")
print(f"[CONFIG] Stride size: {STRIDE_SIZE} samples")


# ==============================================================================
# 2. EXACT PRETRAINED CONVOLUTIONAL BLOCK
#
# IMPORTANT:
#
# The checkpoint contains keys like:
#
#     enc1.block.0.weight
#     enc1.block.1.weight
#     enc1.block.3.weight
#
# Therefore the architecture MUST contain:
#
#     enc1.block
#
# The checkpoint also showed:
#
#     kernel_size = 5
#     bias = False
#
# ==============================================================================

class ConvBlock1D(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels
    ):

        super().__init__()

        self.block = nn.Sequential(

            # ----------------------------------------------------------
            # FIRST CONVOLUTION
            # ----------------------------------------------------------

            nn.Conv1d(
                in_channels=in_channels,
                out_channels=out_channels,
                kernel_size=5,
                padding=2,
                bias=False
            ),

            nn.BatchNorm1d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            ),

            # ----------------------------------------------------------
            # SECOND CONVOLUTION
            # ----------------------------------------------------------

            nn.Conv1d(
                in_channels=out_channels,
                out_channels=out_channels,
                kernel_size=5,
                padding=2,
                bias=False
            ),

            nn.BatchNorm1d(
                out_channels
            ),

            nn.ReLU(
                inplace=True
            )
        )

    def forward(self, x):

        return self.block(x)


# ==============================================================================
# 3. EXACT PRETRAINED CTGUNET1D
#
# 4 INPUT -> 3 OUTPUT
# ==============================================================================

class CTGUNet1D(nn.Module):

    def __init__(
        self,
        in_channels=4,
        out_channels=3
    ):

        super().__init__()


        # ------------------------------------------------------------------
        # Encoder
        # ------------------------------------------------------------------


        self.enc1 = ConvBlock1D(
            in_channels,
            32
          )
        self.pool1 = nn.MaxPool1d(
            2
        )


        self.enc2 = ConvBlock1D(
            32,
            64
        )

        self.pool2 = nn.MaxPool1d(
            2
        )


        self.enc3 = ConvBlock1D(
            64,
            128
        )

        self.pool3 = nn.MaxPool1d(
            2
        )


        # ------------------------------------------------------------------
        # Bottleneck
        # ------------------------------------------------------------------

        self.bottleneck = ConvBlock1D(
            128,
            256
        )


        # ------------------------------------------------------------------
        # Decoder
        # ------------------------------------------------------------------

        self.up3 = nn.ConvTranspose1d(
            256,
            128,
            kernel_size=2,
            stride=2
        )

        self.dec3 = ConvBlock1D(
            256,
            128
        )


        self.up2 = nn.ConvTranspose1d(
            128,
            64,
            kernel_size=2,
            stride=2
        )

        self.dec2 = ConvBlock1D(
            128,
            64
        )


        self.up1 = nn.ConvTranspose1d(
            64,
            32,
            kernel_size=2,
            stride=2
        )

        self.dec1 = ConvBlock1D(
            64,
            32
        )


        # ------------------------------------------------------------------
        # FINAL OUTPUT
        #
        # IMPORTANT:
        # The checkpoint has 3 output channels.
        # ------------------------------------------------------------------

        self.output = nn.Conv1d(
              32,
              out_channels,
              kernel_size=1
        )

    def forward(self, x):

        # ------------------------------------------------------------------
        # Encoder
        # ------------------------------------------------------------------

        c1 = self.enc1(x)

        p1 = self.pool1(
            c1
        )


        c2 = self.enc2(
            p1
        )

        p2 = self.pool2(
            c2
        )


        c3 = self.enc3(
            p2
        )

        p3 = self.pool3(
            c3
        )


        # ------------------------------------------------------------------
        # Bottleneck
        # ------------------------------------------------------------------

        bn = self.bottleneck(
            p3
        )


        # ------------------------------------------------------------------
        # Decoder 3
        # ------------------------------------------------------------------

        u3 = self.up3(
            bn
        )

        if u3.shape[-1] != c3.shape[-1]:

            u3 = nn.functional.interpolate(
                u3,
                size=c3.shape[-1],
                mode="linear",
                align_corners=False
            )

        u3 = torch.cat(
            [u3, c3],
            dim=1
        )

        d3 = self.dec3(
            u3
        )


        # ------------------------------------------------------------------
        # Decoder 2
        # ------------------------------------------------------------------

        u2 = self.up2(
            d3
        )

        if u2.shape[-1] != c2.shape[-1]:

            u2 = nn.functional.interpolate(
                u2,
                size=c2.shape[-1],
                mode="linear",
                align_corners=False
            )

        u2 = torch.cat(
            [u2, c2],
            dim=1
        )

        d2 = self.dec2(
            u2
        )


        # ------------------------------------------------------------------
        # Decoder 1
        # ------------------------------------------------------------------

        u1 = self.up1(
            d2
        )

        if u1.shape[-1] != c1.shape[-1]:

            u1 = nn.functional.interpolate(
                u1,
                size=c1.shape[-1],
                mode="linear",
                align_corners=False
            )

        u1 = torch.cat(
            [u1, c1],
            dim=1
        )

        d1 = self.dec1(
            u1
        )


        # ------------------------------------------------------------------
        # 3-channel output
        # ------------------------------------------------------------------

        return self.output(
            d1
        )


print(
    "[SUCCESS] CTGUNet1D architecture defined."
)

print(
    "[INFO] Pretrained architecture: 4 input -> 3 output"
)

print(
    "[INFO] Conv kernel size: 5"
)

print(
    "[INFO] Conv bias: False"
)

print(
    "[INFO] Output channels: ACC / DEC / BASELINE"
)


# ==============================================================================
# 4. LOAD PRETRAINED CHECKPOINT
# ==============================================================================

print()
print(
    "=============================================================="
)
print(
    "LOADING PRETRAINED DETECTOR"
)
print(
    "=============================================================="
)


loaded_obj = torch.load(
    PRETRAIN_MODEL_PATH,
    map_location=device,
    weights_only=False
)


# ==============================================================================
# 5. EXTRACT STATE DICTIONARY
# ==============================================================================

if isinstance(
    loaded_obj,
    dict
):

    if "model_state_dict" in loaded_obj:

        state_dict = loaded_obj[
            "model_state_dict"
        ]

        print(
            "[INFO] Found 'model_state_dict'."
        )

    elif "state_dict" in loaded_obj:

        state_dict = loaded_obj[
            "state_dict"
        ]

        print(
            "[INFO] Found 'state_dict'."
        )

    else:

        state_dict = loaded_obj

        print(
            "[INFO] Checkpoint appears to be a raw state_dict."
        )

else:

    state_dict = loaded_obj

    print(
        "[INFO] Loaded raw state_dict object."
    )


# ==============================================================================
# 6. HANDLE OPTIONAL "module." PREFIX
#
# This does not change any weights.
# It only removes DataParallel's optional prefix if present.
# ==============================================================================

clean_state_dict = {}

for key, value in state_dict.items():

    if key.startswith(
        "module."
    ):

        clean_key = key[
            len("module.") :
        ]

    else:

        clean_key = key

    clean_state_dict[
        clean_key
    ] = value


state_dict = clean_state_dict


# ==============================================================================
# 7. PRINT CHECKPOINT INFORMATION
# ==============================================================================

print(
    f"[INFO] Number of checkpoint tensors: "
    f"{len(state_dict)}"
)


print(
    "[INFO] Example checkpoint keys:"
)

for i, key in enumerate(
    state_dict.keys()
):

    print(
        f"       {key}"
    )

    if i >= 7:
        break


# ==============================================================================
# 8. CREATE EXACT PRETRAINED MODEL
# ==============================================================================

pretrain = CTGUNet1D(
    in_channels=4,
    out_channels=3
).to(
    device
)


# ==============================================================================
# 9. LOAD CHECKPOINT STRICTLY
#
# If this fails, we STOP here.
#
# We do NOT use strict=False because we do not want to silently load a
# partially incompatible pretrained model.
# ==============================================================================

pretrain.load_state_dict(
    state_dict,
    strict=True
)


print(
    "[SUCCESS] Pretrained checkpoint loaded EXACTLY."
)


# ==============================================================================
# 10. FREEZE PRETRAINED MODEL
# ==============================================================================

pretrain.eval()

for parameter in pretrain.parameters():

    parameter.requires_grad = False


print(
    "[SUCCESS] Pretrained detector is frozen."
)

print(
    "[INFO] Pretrained outputs:"
)

print(
    "[INFO] BASELINE will NOT be passed to the CNN."
)


# ==============================================================================
# 11. LOAD FULL-LENGTH CTG CACHE
# ==============================================================================

cache = np.load(
    FULL_CACHE_PATH,
    allow_pickle=True
)


record_ids = [

    key.replace(
        "rec_",
        ""
    )

    for key in cache.files

    if key.startswith(
        "rec_"
    )
]


print()
print(
    f"[DATA] Found {len(record_ids)} CTG records."
)


# ==============================================================================
# 12. CREATE 6-CHANNEL MIL BAGS
#
# For each valid 15-minute window:
#
#     Original CTG
#          [4, 3600]
#
#             |
#             v
#
#     Pretrained detector
#          [3, 3600]
#
#             |
#             v
#
#       keep channels 0:2
#
#          ACC + DEC
#          [2, 3600]
#
#             |
#             v
#
#       concatenate
#
#          [6, 3600]
#
# ==============================================================================

mil_bags = {}


#dropped_count = 0
#total_windows = 0
#valid_windows = 0


# ==============================================================================
# CREATE MIL BAGS
# ==============================================================================

mil_bags = create_mil_bags(
    cache_path=FULL_CACHE_PATH,
    pretrain_model=pretrain if CONFIG["use_pretraining"] else None,
    n_channels=CONFIG["n_channels"],
    window_size=WINDOW_SIZE,
    stride_size=STRIDE_SIZE,
    max_allowable_nan_pct=MAX_ALLOWABLE_NAN_PCT,
    device=device
)

print(
    f"[SUCCESS] Created MIL bags for {len(mil_bags)} records."
)

# ==============================================================================
# 13. MIL BAG SUMMARY
# ==============================================================================

print()
print(
    "=============================================================="
)

print(
    "6-CHANNEL MIL BAG CREATION COMPLETE"
)

print(
    "=============================================================="
)

print(
    f"Records found:           {len(record_ids)}"
)

print(
    f"Records with valid bags: {len(mil_bags)}"
)

print(
    "=============================================================="
)


# ==============================================================================
# 14. EXISTING CNN
#
# ONLY CHANGE:
#
#     4 input channels -> 6 input channels
#
# Everything after the first Conv1d remains unchanged.
# ==============================================================================


class CNNEmbeddingExtractor(nn.Module):

    def __init__(
        self,
        in_channels,
        embedding_dim=16,
        n_kernels=16
    ):

        super().__init__()

        c1 = n_kernels
        c2 = n_kernels * 2
        c3 = n_kernels * 4

        self.conv = nn.Sequential(

            nn.Conv1d(
                in_channels,
                c1,
                kernel_size=7,
                padding=3
            ),

            nn.BatchNorm1d(c1),
            nn.ReLU(),
            nn.MaxPool1d(4),

            nn.Conv1d(
                c1,
                c2,
                kernel_size=5,
                padding=2
            ),

            nn.BatchNorm1d(c2),
            nn.ReLU(),
            nn.MaxPool1d(4),

            nn.Conv1d(
                c2,
                c3,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm1d(c3),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1)
        )

        self.project = nn.Linear(
            c3,
            embedding_dim
        )

    def forward(self, x):

        features = self.conv(x).squeeze(-1)

        return self.project(features)

# ==============================================================================
# 15. CHUNK DATASET
# ==============================================================================

class ChunkDataset(
    Dataset
):

    def __init__(
        self,
        chunks_array
    ):

        self.chunks = (

            torch.from_numpy(
                chunks_array
            )
            .float()

        )


    def __len__(
        self
    ):

        return len(
            self.chunks
        )


    def __getitem__(
        self,
        idx
    ):

        return self.chunks[
            idx
        ]


# ==============================================================================
# 16. INITIALIZE CNN
# ==============================================================================

cnn_extractor = CNNEmbeddingExtractor(
    in_channels=CONFIG["n_channels"],
    embedding_dim=16
).to(device)

cnn_extractor.eval()


print(
    "[SUCCESS] CNN initialized."
)


print(
    f"[INFO] CNN expects: "
    f"[Batch, {CONFIG['n_channels']}, {WINDOW_SIZE}]"
)

# ==============================================================================
# 17. EXTRACT 16-D INSTANCE EMBEDDINGS
# ==============================================================================

cnn_extractor.eval()
record_embeddings = {}

with torch.no_grad():

    for rid, chunks in mil_bags.items():


        dataset = ChunkDataset(
            chunks
        )


        loader = DataLoader(
            dataset,
            batch_size=16,
            shuffle=False
        )


        embeddings_list = []


        for batch_chunks in loader:


            batch_chunks = (
                batch_chunks.to(device)
            )


            embeddings = (
                cnn_extractor(
                    batch_chunks
                )
            )


            embeddings_list.append(

                embeddings
                .cpu()
                .numpy()

            )


        record_embeddings[
            rid
        ] = np.concatenate(
            embeddings_list,
            axis=0
        )


# ==============================================================================
# 18. FINAL VERIFICATION
# ==============================================================================

print()
print(
    "=============================================================="
)

print(
    "PIPELINE VERIFICATION"
)

print(
    "=============================================================="
)


if len(
    mil_bags
) > 0:


    first_rid = next(
        iter(mil_bags)
    )


    print(
        f"Example record ID:       {first_rid}"
    )

    print(
    f"{CONFIG['n_channels']}-channel bag shape: "
    f"{mil_bags[first_rid].shape}"
    )

    print(
        "16-D embedding bag:      "
        f"{record_embeddings[first_rid].shape}"
    )


    print()
    print(
        "Expected:"
    )


    print(
        f"    {CONFIG['n_channels']}-channel bag:"
    )

    print(
        f"    [N_valid_windows, {CONFIG['n_channels']}, {WINDOW_SIZE}]"
    )


    print(
        "    16-D embedding bag:"
    )

    print(
        "    [N_valid_windows, 16]"
    )


else:

    print(
        "[WARNING] No valid MIL bags were created."
    )


# ==============================================================================
# 19. FINAL STATUS
# ==============================================================================

print()
print(
    "=============================================================="
)

print(
    "FINAL PIPELINE STATUS"
)

print(
    "=============================================================="
)

print(
    "[SUCCESS] Pretrained detector = 4 input -> 3 output."
)

print(
    "[SUCCESS] Pretrained architecture uses kernel size 5."
)

print(
    "[SUCCESS] Pretrained convolution biases = False."
)

print(
    "[SUCCESS] Checkpoint loaded strictly."
)

print(
    "[SUCCESS] ACC channel retained."
)

print(
    "[SUCCESS] DEC channel retained."
)

print(
    "[SUCCESS] BASELINE channel discarded."
)

print(
    "[SUCCESS] Original CTG = 4 channels."
)

print(
    "[SUCCESS] Additional ACC + DEC = 2 channels."
)

print(
    "[SUCCESS] Final CNN input = 6 channels."
)

print(
    "[SUCCESS] Pretrained detector is frozen."
)

print(
    "[SUCCESS] No attention mechanism added."
)

print(
    "[SUCCESS] No MIL architecture redesign."
)

print(
    "[SUCCESS] Only CNN input changed from 4 -> 6."
)

print(
    "=============================================================="
)

[DEVICE] Using: cpu
[CONFIG] Window size: 3600 samples
[CONFIG] Stride size: 1200 samples
[SUCCESS] CTGUNet1D architecture defined.
[INFO] Pretrained architecture: 4 input -> 3 output
[INFO] Conv kernel size: 5
[INFO] Conv bias: False
[INFO] Output channels: ACC / DEC / BASELINE

LOADING PRETRAINED DETECTOR
[INFO] Found 'model_state_dict'.
[INFO] Number of checkpoint tensors: 92
[INFO] Example checkpoint keys:
       enc1.block.0.weight
       enc1.block.1.weight
       enc1.block.1.bias
       enc1.block.1.running_mean
       enc1.block.1.running_var
       enc1.block.1.num_batches_tracked
       enc1.block.3.weight
       enc1.block.4.weight
[SUCCESS] Pretrained checkpoint loaded EXACTLY.
[SUCCESS] Pretrained detector is frozen.
[INFO] Pretrained outputs:
[INFO] BASELINE will NOT be passed to the CNN.

[DATA] Found 546 CTG records.
[SUCCESS] Created MIL bags for 546 records.

6-CHANNEL MIL BAG CREATION COMPLETE
Records found:           546
Records with valid bags: 546
[SUCCESS] CNN i

In [5]:
# ==============================================================================
# SAVE 6-CHANNEL MIL BAGS + 16-D CHUNK EMBEDDINGS
# ==============================================================================

import os
import pickle

SAVE_DIR = (
    "/content/drive/My Drive/Colab Notebooks/Fetal_project/"
)

BAGS_SAVE_PATH = os.path.join(
    SAVE_DIR,
    f"ctu_{CONFIG['n_channels']}channel_mil_bags.pkl"
)

EMBEDDINGS_SAVE_PATH = os.path.join(
    SAVE_DIR,
    f"ctu_{CONFIG['n_channels']}channel_chunk_embeddings.pkl"
)


# ------------------------------------------------------------------------------
# Save 6-channel bags
# ------------------------------------------------------------------------------

with open(BAGS_SAVE_PATH, "wb") as f:
    pickle.dump(
        mil_bags,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )


print(
    f"[SUCCESS] {CONFIG['n_channels']}-channel MIL bags saved:"
)

print(
    BAGS_SAVE_PATH
)


# ------------------------------------------------------------------------------
# Save 16-D chunk embeddings
# ------------------------------------------------------------------------------

with open(EMBEDDINGS_SAVE_PATH, "wb") as f:
    pickle.dump(
        record_embeddings,
        f,
        protocol=pickle.HIGHEST_PROTOCOL
    )


print(
    "[SUCCESS] 16-D chunk embeddings saved:"
)

print(
    EMBEDDINGS_SAVE_PATH
)


# ------------------------------------------------------------------------------
# Verification
# ------------------------------------------------------------------------------

print()
print("==============================================================")
print("SAVE VERIFICATION")
print("==============================================================")

print(
    f"Number of bags:       {len(mil_bags)}"
)

print(
    f"Number of embeddings: {len(record_embeddings)}"
)


if len(record_embeddings) > 0:

    first_rid = next(
        iter(record_embeddings)
    )

    print(
        f"Example record:       {first_rid}"
    )

    print(
        f"Bag shape:             "
        f"{mil_bags[first_rid].shape}"
    )

    print(
        f"Embedding shape:       "
        f"{record_embeddings[first_rid].shape}"
    )


print("==============================================================")
print("[SUCCESS] Preprocessing results safely saved to Google Drive.")
print("==============================================================")

[SUCCESS] 4-channel MIL bags saved:
/content/drive/My Drive/Colab Notebooks/Fetal_project/ctu_4channel_mil_bags.pkl
[SUCCESS] 16-D chunk embeddings saved:
/content/drive/My Drive/Colab Notebooks/Fetal_project/ctu_4channel_chunk_embeddings.pkl

SAVE VERIFICATION
Number of bags:       546
Number of embeddings: 546
Example record:       1001
Bag shape:             (14, 4, 3600)
Embedding shape:       (14, 16)
[SUCCESS] Preprocessing results safely saved to Google Drive.
